# Домашнее задание 7
## Задача выведения КА на орбиту

**Введение в маневрирование КА · К лекции 10**  
**Срок сдачи: 30 ноября 2026 года, 23:59.**

Рассчитайте характеристическую скорость двухступенчатой ракеты и исследуйте,
как широта космодрома связана с начальной скоростью и направлением запуска.

**Оба задания обязательны.**


### Как работать с ноутбуком

Выполняйте ячейки по порядку. Нужны Python 3, NumPy и Matplotlib.
Готовые функции выводят таблицы, строят графики и позволяют при желании проверить ответы;
их код менять не требуется. Самостоятельно получите формулы, заполните
переменные со значением `None` и текстовые ячейки с ответами.

**Чекеры предназначены только для самопроверки. Использовать их не обязательно.**
Ячейки с их вызовами можно пропустить; сохранять результаты проверок не нужно.

**Единицы и порядок компонент:**
- в расчётах массы — в **килограммах**, тяга — в **ньютонах**, время — в **секундах**;
- длины — в **метрах**, скорости — в **м/с**, углы — в **радианах**;
- готовые таблицы переводят массы в тонны, азимуты и широты — в градусы;
- `masses_1` содержит четыре массы: старт, до разделения, после разделения, конец;
- в `burn_times_1` и `dv_stages_1` первая строка массива относится к первой ступени,
  вторая — к верхней;
- функции истории получают **скалярное** время от старта; сетки для рисунков строятся автоматически;
- в момент разделения `mass_history_1(t)` возвращает массу **после** отделения;
- в задании 2 строки соответствуют широтам $0^\circ,28{,}5^\circ,45^\circ,60^\circ$;
- столбцы `azimuths_2` — меньший и больший азимуты в $[0,\pi]$;
- недоступной широте соответствуют `False` в `reachable_2` и **два `np.nan`** в её строке азимутов.

Не округляйте промежуточные результаты. Численные проверки не заменяют
выводы и объяснения. Пока ответы не заполнены, ячейки выводят напоминания
и не прерывают выполнение ноутбука.


## Исходные данные

Масса полезной нагрузки — $5$ т. Характеристики ракеты:

| Параметр | Первая ступень | Верхняя ступень |
|---|---:|---:|
| Масса топлива, т | 160 | 45 |
| Сухая масса, т | 16 | 4 |
| Тяга, МН | 3,5 | 0,8 |
| Удельный импульс, с | 300 | 450 |

Примите $g_0=9{,}80665$ м/с². Тяга и удельный импульс постоянны.
Первая ступень расходует всё топливо и отделяется; сразу после этого включается
верхняя ступень, которая также расходует **всё** топливо.
Пассивной паузы между ступенями нет. Конечная масса включает сухую верхнюю
ступень и полезную нагрузку. В отличие от расчётного примера лекции,
остаток топлива верхней ступени здесь равен нулю.

Для геометрической оценки запуска используйте сферическую Землю:
$$R_E=6378{,}137\ \text{км},\qquad
  \omega_E=7{,}2921159\cdot10^{-5}\ \text{рад/с}.$$
Целевое наклонение $i=51{,}6^\circ$;
широты космодромов $\phi=0^\circ,28{,}5^\circ,45^\circ,60^\circ$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

ACCENT = "#187AA5"
SECOND = "#A36540"
INK = "#302B28"
fonts = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams.update({
    "font.family": "PT Serif" if "PT Serif" in fonts else "DejaVu Serif",
    "font.size": 11,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})


In [ ]:
G0 = 9.80665                        # м/с²
M_PAYLOAD = 5_000.0                 # кг
M_PROP = np.array([160_000., 45_000.])  # кг: первая, верхняя
M_DRY = np.array([16_000., 4_000.])    # кг: первая, верхняя
THRUST = np.array([3.5e6, 0.8e6])   # Н: первая, верхняя
ISP = np.array([300., 450.])         # с: первая, верхняя

R_E = 6_378_137.0                   # м
OMEGA_E = 7.2921159e-5              # рад/с
I_TARGET = np.radians(51.6)          # рад
LATITUDES = np.radians([0., 28.5, 45., 60.])
PHI_PLOT_MAX = np.radians(70.)

print("Исходные данные заданы в СИ. Углы в расчётах — в радианах.")


### Готовые функции для таблиц, графиков и самопроверки

Выполните следующие ячейки. Разбирать их код для выполнения задания не нужно.

`mass_table(masses)` выводит массы на четырёх границах;
`burn_table(burn_times, dv_stages, dv_total)` — времена и характеристические скорости.
`comparison_table(...)` сравнивает отделение и сохранение сухой первой ступени.

`plot_histories(mass_function, char_function, masses, burn_times)` строит
на общей шкале $m(t)$ и накопленную $\Delta V_{\mathrm{th}}(t)$.
Она сама создаёт сетки до и после разделения, отмечает момент разделения,
отдельно показывает массу перед сбросом и после него.
Передайте **имена функций**, без скобок и аргументов.

`launch_table(v_east, reachable, azimuths)` выводит результаты для четырёх космодромов.
Сначала достаточно заполнить начальные скорости; после расчёта доступности
и азимутов таблица дополняется остальными столбцами.
`plot_azimuths(branch_function, phi_boundary)` строит обе ветви азимута
от широты, отмечает границу доступности и область за ней.
Функция студента получает одну широту в радианах и возвращает
пару азимутов в радианах либо `(np.nan, np.nan)` при отсутствии решения.

`check_task1(...)` и `check_task2(...)` возвращают `True` при успешной
численной проверке и `False`, если обнаружена ошибка.
Если ответы ещё не заполнены, возвращается `None`.


In [ ]:
def _answers_ready(*pairs):
    """Проверка формы и конечности числовых ответов перед выводом."""
    for value, shape in pairs:
        if value is None:
            return False
        try:
            array = np.asarray(value)
        except (TypeError, ValueError):
            return False
        if (array.shape != shape or array.dtype.kind not in "biuf"
                or not np.all(np.isfinite(array))):
            return False
    return True


def _history_values(function, times, name):
    if not callable(function):
        print(f"Нужна функция {name}, принимающая одно скалярное время.")
        return None
    values = []
    try:
        for t in times:
            result = function(float(t))
            if result is None:
                print(f"Сначала заполните функцию {name}.")
                return None
            if not _answers_ready((result, ())):
                print(f"{name} должна возвращать одно конечное число для каждого времени.")
                return None
            values.append(float(result))
    except Exception as error:
        print(f"Не удалось вычислить {name}: {type(error).__name__}: {error}")
        return None
    return np.asarray(values)


def mass_table(masses):
    names = ("Старт", "До разделения", "После разделения", "Конец")
    print(f"{'Состояние':<22} {'Масса, т':>15}")
    for name, mass in zip(names, masses):
        print(f"{name:<22} {mass / 1000:15.6f}")


def burn_table(burn_times, dv_stages, dv_total):
    print(f"{'Ступень':<15} {'Работа, с':>15} {'ΔV_th, м/с':>18}")
    for name, time, dv in zip(("Первая", "Верхняя"), burn_times, dv_stages):
        print(f"{name:<15} {time:15.6f} {dv:18.6f}")
    print(f"Суммарная характеристическая скорость: {float(dv_total):.6f} м/с")


def comparison_table(dv_upper_no_sep, dv_total_no_sep, dv_stages, dv_total):
    print(f"{'Случай':<26} {'Верхняя, м/с':>18} {'Сумма, м/с':>18}")
    print(f"{'С отделением':<26} {dv_stages[1]:18.6f} {float(dv_total):18.6f}")
    print(f"{'Без отделения':<26} {float(dv_upper_no_sep):18.6f} {float(dv_total_no_sep):18.6f}")
    print(f"Разность суммарных значений: {float(dv_total - dv_total_no_sep):.6f} м/с")


def plot_histories(mass_function, char_function, masses, burn_times):
    if (not _answers_ready((masses, (4,)), (burn_times, (2,)))
            or np.any(np.asarray(masses) <= 0) or np.any(np.asarray(burn_times) <= 0)):
        print("Для графиков нужны четыре положительные массы и две положительные длительности.")
        return None
    masses = np.asarray(masses, dtype=float)
    tau1, tau2 = np.asarray(burn_times, dtype=float)
    # Две отдельные сетки не соединяют скачок массы наклонным отрезком.
    first = np.linspace(0., tau1, 301, endpoint=False)
    second = np.linspace(tau1, tau1 + tau2, 401)
    m_first = _history_values(mass_function, first, "mass_history_1")
    m_second = _history_values(mass_function, second, "mass_history_1")
    dv_first = _history_values(char_function, np.r_[first, tau1], "char_speed_history_1")
    dv_second = _history_values(char_function, second, "char_speed_history_1")
    if any(value is None for value in (m_first, m_second, dv_first, dv_second)):
        return None
    fig, axes = plt.subplots(2, 1, figsize=(10, 7.2), sharex=True, constrained_layout=True)
    axes[0].plot(np.r_[first, tau1], np.r_[m_first, masses[1]] / 1000,
                 color=ACCENT, linewidth=2, label="Первая ступень")
    axes[0].plot(second, m_second / 1000, color=SECOND, linewidth=2,
                 label="Верхняя ступень")
    axes[0].plot([tau1, tau1], masses[1:3] / 1000, color=INK,
                 linestyle=":", linewidth=1.2)
    axes[0].scatter(tau1, masses[1] / 1000, s=38, edgecolor=ACCENT,
                    facecolor="white", zorder=4)
    axes[0].scatter(tau1, m_second[0] / 1000, s=38, color=SECOND, zorder=4)
    axes[1].plot(np.r_[first, tau1], dv_first, color=ACCENT, linewidth=2)
    axes[1].plot(second, dv_second, color=SECOND, linewidth=2)
    for ax in axes:
        ax.axvline(tau1, color=INK, linestyle="--", linewidth=1.0,
                   label="Разделение")
        ax.set_xlim(0., tau1 + tau2)
        ax.grid(alpha=0.22)
    axes[0].set(ylabel="Масса, т", title="Задание 1. Масса и характеристическая скорость")
    axes[1].set(xlabel="Время от старта, с", ylabel=r"$\Delta V_{\rm th}$, м/с")
    axes[0].legend(loc="best")
    return fig, axes


def _azimuth_array_ready(azimuths, reachable):
    try:
        values, mask = np.asarray(azimuths), np.asarray(reachable)
    except (TypeError, ValueError):
        return False
    if (values.shape != (4, 2) or values.dtype.kind not in "iuf"
            or mask.shape != (4,) or mask.dtype.kind != "b"):
        return False
    return (bool(np.all(np.isfinite(values[mask])))
            and bool(np.all(np.isnan(values[~mask]))))


def _branch_values(function, latitudes):
    if not callable(function):
        print("Нужна функция azimuth_branches_2, принимающая одну широту в радианах.")
        return None
    rows = []
    try:
        for latitude in latitudes:
            result = function(float(latitude))
            if result is None:
                print("Сначала заполните функцию azimuth_branches_2.")
                return None
            pair = np.asarray(result)
            if (pair.shape != (2,) or pair.dtype.kind not in "iuf"
                    or not (np.all(np.isfinite(pair)) or np.all(np.isnan(pair)))):
                print("azimuth_branches_2 должна возвращать два числа либо два np.nan.")
                return None
            rows.append(pair.astype(float))
    except Exception as error:
        print(f"Не удалось вычислить azimuth_branches_2: {type(error).__name__}: {error}")
        return None
    return np.asarray(rows)


def launch_table(v_east, reachable=None, azimuths=None):
    if not _azimuth_array_ready(azimuths, reachable):
        print(f"{'Широта, °':>12} {'V_E,0, м/с':>16}")
        for latitude, speed in zip(LATITUDES, v_east):
            print(f"{np.degrees(latitude):12.1f} {speed:16.6f}")
        print("Пункт 2.2: для добавления азимутов заполните reachable_2 и azimuths_2. "
              "Нужны четыре булевых признака и массив (4, 2); "
              "для недоступных широт — два np.nan.")
        return
    print(f"{'Широта, °':>12} {'V_E,0, м/с':>16} {'Доступно':>12} {'A₁, °':>14} {'A₂, °':>14}")
    for latitude, speed, valid, pair in zip(LATITUDES, v_east, reachable, azimuths):
        angles = [f"{np.degrees(a):14.6f}" for a in pair] if valid else [f"{'—':>14}"] * 2
        print(f"{np.degrees(latitude):12.1f} {speed:16.6f} {str(bool(valid)):>12} "
              + " ".join(angles))


def plot_azimuths(branch_function, phi_boundary):
    if (not _answers_ready((phi_boundary, ()))
            or not 0 <= float(phi_boundary) <= PHI_PLOT_MAX):
        print("Для графика укажите phi_boundary_2 в радианах внутри заданного диапазона широт.")
        return None
    boundary = float(phi_boundary)
    latitudes = np.unique(np.r_[np.linspace(0., PHI_PLOT_MAX, 701), boundary])
    values = _branch_values(branch_function, latitudes)
    if values is None:
        return None
    fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    for j, color in enumerate((ACCENT, SECOND)):
        ax.plot(np.degrees(latitudes), np.degrees(values[:, j]),
                color=color, linewidth=2, label=fr"Ветвь $A_{j + 1}$")
    ax.axvline(np.degrees(boundary), color=INK, linestyle="--", linewidth=1.2,
               label="Граница доступности")
    ax.axvspan(np.degrees(boundary), np.degrees(PHI_PLOT_MAX),
               color=ACCENT, alpha=0.08, label="За границей доступности")
    ax.set(xlabel=r"Широта $\phi$, град", ylabel=r"Инерциальный азимут $A$, град",
           xlim=(0., np.degrees(PHI_PLOT_MAX)), ylim=(0., 180.),
           title="Задание 2. Азимут запуска и широта космодрома")
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    return fig, ax


In [ ]:
def _report(title, checks):
    print(title)
    for passed, message in checks:
        print(("[OK] " if passed else "[Проверьте] ") + message)
    passed = all(bool(item[0]) for item in checks)
    print("Численные проверки пройдены." if passed
          else "Есть замечания; исправьте отмеченные пункты и повторите проверку.")
    return passed


def check_task1(masses, burn_times, dv_stages, dv_total,
                mass_function, char_function, dv_upper_no_sep, dv_total_no_sep):
    values = (masses, burn_times, dv_stages, dv_total, dv_upper_no_sep, dv_total_no_sep)
    if any(value is None for value in values):
        print("Задание 1: сначала заполните массы, длительности и характеристические скорости обоих случаев.")
        return None
    if not _answers_ready((masses, (4,)), (burn_times, (2,)), (dv_stages, (2,)),
                          (dv_total, ()), (dv_upper_no_sep, ()), (dv_total_no_sep, ())):
        print("Проверьте формы массивов и конечность чисел. Массы — кг, скорости — м/с.")
        return False
    masses, burn_times, dv_stages = [np.asarray(v, dtype=float)
                                    for v in (masses, burn_times, dv_stages)]
    c = G0 * ISP
    expected_masses = np.array([M_PAYLOAD + M_PROP.sum() + M_DRY.sum(),
                               M_PAYLOAD + M_PROP[1] + M_DRY.sum(),
                               M_PAYLOAD + M_PROP[1] + M_DRY[1],
                               M_PAYLOAD + M_DRY[1]])
    expected_times = c * M_PROP / THRUST
    expected_dv = c * np.log(expected_masses[[0, 2]] / expected_masses[[1, 3]])
    expected_upper_no_sep = c[1] * np.log(expected_masses[1]
                                          / (expected_masses[3] + M_DRY[0]))
    tau1, tau2 = expected_times
    eps = min(tau1, tau2) * 1e-7
    times = np.r_[np.linspace(0., tau1, 7, endpoint=False), tau1 - eps,
                  tau1, tau1 + eps, np.linspace(tau1, tau1 + tau2, 7)]
    sampled_m = _history_values(mass_function, times, "mass_history_1")
    sampled_dv = _history_values(char_function, times, "char_speed_history_1")
    if sampled_m is None or sampled_dv is None:
        print("Для полной самопроверки нужны обе заполненные функции истории.")
        return None
    first = times < tau1
    expected_m = np.where(first,
                          expected_masses[0] - THRUST[0] / c[0] * times,
                          expected_masses[2] - THRUST[1] / c[1] * (times - tau1))
    expected_history_dv = np.empty_like(times)
    expected_history_dv[first] = c[0] * np.log(expected_masses[0] / expected_m[first])
    expected_history_dv[~first] = expected_dv[0] + c[1] * np.log(
        expected_masses[2] / expected_m[~first])
    checks = [
        (np.allclose(masses, expected_masses, rtol=1e-8, atol=1e-4),
         "Четыре массы; сухая нижняя ступень входит в массу до отделения."),
        (np.allclose(burn_times, expected_times, rtol=1e-7, atol=1e-5),
         "Время работы каждой ступени; расход топлива и тяга заданы в СИ."),
        (np.allclose(dv_stages, expected_dv, rtol=1e-6, atol=1e-4),
         "Характеристические скорости ступеней; для верхней взята масса после отделения."),
        (np.isclose(dv_total, expected_dv.sum(), rtol=1e-6, atol=1e-4),
         "Суммарная характеристическая скорость в м/с."),
        (np.allclose(sampled_m, expected_m, rtol=1e-8, atol=1e-4),
         "История массы, включая левую окрестность разделения, само разделение и конец."),
        (np.allclose(sampled_dv, expected_history_dv, rtol=1e-6, atol=1e-4),
         "Накопленная характеристическая скорость непрерывна при отделении сухой массы."),
        (np.isclose(dv_upper_no_sep, expected_upper_no_sep, rtol=1e-6, atol=1e-4),
         "Верхняя ступень без отделения нижней: сухая масса нижней остаётся до конца."),
        (np.isclose(dv_total_no_sep, expected_dv[0] + expected_upper_no_sep,
                    rtol=1e-6, atol=1e-4),
         "Суммарная характеристическая скорость без отделения; первая ступень не меняется."),
    ]
    return _report("Самопроверка задания 1", checks)


def check_task2(v_east, reachable, azimuths, phi_boundary, branch_function, i_east):
    if any(value is None for value in (v_east, reachable, azimuths, phi_boundary, i_east)):
        print("Задание 2: сначала заполните скорости, доступность, азимуты, границу и наклонения восточного запуска.")
        return None
    if not _answers_ready((v_east, (4,)), (phi_boundary, ()), (i_east, (4,))):
        print("Проверьте числовые массивы: четыре скорости, четыре наклонения и одна граница.")
        return False
    reachable = np.asarray(reachable)
    if reachable.shape != (4,) or reachable.dtype.kind != "b":
        print("reachable_2 должен содержать четыре булевых значения True/False.")
        return False
    if not _azimuth_array_ready(azimuths, reachable):
        print("azimuths_2: массив (4, 2), числа для доступных широт и два np.nan для недоступных.")
        return False
    azimuths = np.asarray(azimuths, dtype=float)
    expected_speed = OMEGA_E * R_E * np.cos(LATITUDES)
    ratio = np.cos(I_TARGET) / np.cos(LATITUDES)
    expected_reachable = np.abs(ratio) <= 1 + 1e-12
    expected_azimuths = np.full((4, 2), np.nan)
    a = np.arcsin(np.clip(ratio[expected_reachable], -1., 1.))
    expected_azimuths[expected_reachable] = np.column_stack((a, np.pi - a))
    # Для выбранных северных широт и прямой орбиты граница равна целевому i.
    boundary = I_TARGET
    probes = np.r_[0., np.radians(12.3), LATITUDES, boundary - 1e-5,
                   boundary, boundary + 1e-5, PHI_PLOT_MAX]
    sampled = _branch_values(branch_function, probes)
    if sampled is None:
        print("Для полной самопроверки заполните функцию azimuth_branches_2.")
        return None
    reference = np.full((len(probes), 2), np.nan)
    ratio_probes = np.cos(I_TARGET) / np.cos(probes)
    valid = np.abs(ratio_probes) <= 1 + 1e-12
    values = np.arcsin(np.clip(ratio_probes[valid], -1., 1.))
    reference[valid] = np.column_stack((values, np.pi - values))
    checks = [
        (np.allclose(v_east, expected_speed, rtol=1e-6, atol=1e-6),
         "Начальные восточные скорости; радиус Земли переведён в метры."),
        (np.array_equal(reachable, expected_reachable),
         "Геометрическая доступность целевого наклонения для всех четырёх широт."),
        (np.allclose(azimuths, expected_azimuths, rtol=1e-6, atol=1e-7, equal_nan=True),
         "Оба инерциальных азимута в радианах, от меньшего к большему; недоступный случай — NaN."),
        (np.isclose(phi_boundary, boundary, rtol=1e-7, atol=1e-7),
         "Граница доступных северных широт; единицы — радианы."),
        (np.allclose(sampled, reference, rtol=1e-6, atol=1e-7, equal_nan=True),
         "Обе ветви на промежуточных широтах, в точке их слияния и за границей."),
        (np.allclose(i_east, LATITUDES, rtol=1e-7, atol=1e-7),
         "Наклонения прямого запуска на восток для заданных северных широт."),
    ]
    return _report("Самопроверка задания 2", checks)


## Задание 1. Характеристическая скорость двухступенчатой ракеты

1. Найдите массу при старте, перед разделением, после разделения
   и после окончания работы верхней ступени.
2. Рассчитайте длительности работы двигателей, характеристические скорости
   обеих ступеней и их сумму.
3. Постройте $m(t)$ и накопленную $\Delta V_{\mathrm{th}}(t)$.
   Отметьте разделение и объясните, какая из величин испытывает скачок.
4. Найдите суммарную характеристическую скорость, если сухая первая ступень
   **не отделяется**. Сохраните остальные данные. Сравните результаты
   и объясните выигрыш от отделения без скачка скорости ракеты.

**Подсказка.** На каждом активном участке
$$c=g_0I_{\mathrm{sp}},\qquad \dot m=-F/c,\qquad
  \Delta V_{\mathrm{th}}=c\ln(m_b/m_e).$$
Для верхней ступени аккуратно определите обе массы в логарифме.
При построении накопленной характеристической скорости на втором участке сохраните
результат, набранный на первом. Сброс сухой массы не является работой двигателя.


**Массовый состав, длительности и характеристические скорости:**

*Запишите, какие части ракеты входят в каждую массу. Приведите расчётные
формулы длительностей и характеристических скоростей, затем заполните ячейку.*


In [ ]:
masses_1 = None       # массив (4,), кг: старт, до разделения, после разделения, конец
burn_times_1 = None   # массив (2,), с: первая и верхняя ступени
dv_stages_1 = None    # массив (2,), м/с: ΔV_th первой и верхней ступеней
dv_total_1 = None     # сумма характеристических скоростей, м/с


In [ ]:
if _answers_ready((masses_1, (4,))):
    mass_table(np.asarray(masses_1, dtype=float))
else:
    print("Пункт 1.1: заполните masses_1 четырьмя конечными числами в килограммах.")

if _answers_ready((burn_times_1, (2,)), (dv_stages_1, (2,)), (dv_total_1, ())):
    print()
    burn_table(np.asarray(burn_times_1, dtype=float),
               np.asarray(dv_stages_1, dtype=float), dv_total_1)
else:
    print("Пункт 1.2: заполните burn_times_1, dv_stages_1 и dv_total_1 для второй таблицы.")


**Истории массы и накопленной характеристической скорости:**

*Запишите кусочные формулы для двух последовательных активных участков.
Время $t$ отсчитывается от старта и не обнуляется при включении верхней ступени.
В самой точке разделения функция массы возвращает значение после сброса.
Функция накопленной характеристической скорости должна быть непрерывной.*

Заполните две функции ниже. Достаточно обычного условия `if` для скалярного `t`;
векторизацию и сетки для графиков писать не требуется.


In [ ]:
def mass_history_1(t):
    """Масса, кг, при скалярном t от 0 до суммы длительностей; в t=τ1 — после разделения."""
    return None


def char_speed_history_1(t):
    """Накопленная характеристическая скорость, м/с, при скалярном времени от старта."""
    return None


In [ ]:
result_1 = plot_histories(mass_history_1, char_speed_history_1, masses_1, burn_times_1)
if result_1 is not None:
    fig, axes = result_1
    plt.show()


**Сохранение отработавшей первой ступени:**

*Повторите расчёт верхнего участка, оставив сухую нижнюю ступень
в составе ракеты до конца. Запишите обе массы в логарифме.
Длительности и результат первой ступени сохраняются.*


In [ ]:
dv_upper_no_sep_1 = None  # характеристическая скорость верхней ступени без отделения нижней, м/с
dv_total_no_sep_1 = None  # сумма по обеим ступеням для этого случая, м/с


In [ ]:
if _answers_ready((dv_upper_no_sep_1, ()), (dv_total_no_sep_1, ()),
                  (dv_stages_1, (2,)), (dv_total_1, ())):
    comparison_table(dv_upper_no_sep_1, dv_total_no_sep_1,
                     np.asarray(dv_stages_1, dtype=float), dv_total_1)
else:
    print("Для сравнения заполните характеристические скорости с отделением и без него.")


In [ ]:
check_task1(masses_1, burn_times_1, dv_stages_1, dv_total_1,
             mass_history_1, char_speed_history_1, dv_upper_no_sep_1, dv_total_no_sep_1)


**Объяснение результатов задания 1:**

*Какая величина меняется скачком при отделении, а какая непрерывна?
Почему сброс сухой массы повышает возможности верхней ступени,
хотя сам по себе не сообщает ракете импульс?*


## Задание 2. Широта космодрома и азимут запуска

Требуется прямое выведение на орбиту с наклонением $i=51{,}6^\circ$
без манёвра изменения плоскости. Используйте геометрическую оценку из лекции.
Азимут $A$ — азимут **инерциальной горизонтальной скорости**, отсчитываемый
от севера к востоку. Рассмотрите четыре широты из `LATITUDES`.

1. Найдите модуль начальной инерциальной скорости каждой площадки в м/с
   и укажите её направление.
2. Из $\cos i=\cos\phi\sin A$ получите условие существования
   допустимого запуска. Найдите все $0\leq A\leq\pi$ либо укажите отсутствие
   решения. Представьте таблицу.
3. Постройте обе ветви $A(\phi)$ при $0^\circ\leq\phi\leq70^\circ$.
   Недопустимые широты пропускайте и отметьте границу области существования.
4. Объясните различие двух азимутов. Какое наклонение получается при запуске
   строго на восток и почему оно минимально для прямого запуска с данной широты?

**Подсказка.** Начальная скорость площадки определяется
$\boldsymbol V_0=\boldsymbol\omega_E\times\boldsymbol R_0$;
расстояние до оси вращения равно $R_E\cos\phi$.
Перед вычислением обратной тригонометрической функции проверьте область её определения.
Одно значение `arcsin` не даёт всех решений уравнения для синуса.
Небольшую погрешность аргумента на самой границе можно устранить `np.clip`.


**Начальные скорости, условие доступности и обе ветви азимута:**

*Получите формулы, укажите область доступных широт и направление начальной
скорости площадки. Покажите, как восстанавливается вторая ветвь азимута.*


In [ ]:
v_east_2 = None        # массив (4,), м/с: начальные восточные скорости площадок
reachable_2 = None     # массив (4,), строго булевы значения True/False
azimuths_2 = None      # массив (4,2), рад: меньший и больший A; два np.nan, если запуска нет
phi_boundary_2 = None  # граница доступных северных широт, рад
i_east_2 = None        # массив (4,), рад: наклонение при запуске строго на восток


def azimuth_branches_2(phi):
    """Два отсортированных азимута, рад, для одной широты; вне области — (np.nan, np.nan)."""
    return None


In [ ]:
if _answers_ready((v_east_2, (4,))):
    launch_table(np.asarray(v_east_2, dtype=float), reachable_2, azimuths_2)
else:
    print("Пункт 2.1: заполните v_east_2 четырьмя конечными скоростями площадок в м/с.")


In [ ]:
result_2 = plot_azimuths(azimuth_branches_2, phi_boundary_2)
if result_2 is not None:
    fig, ax = result_2
    plt.show()


In [ ]:
check_task2(v_east_2, reachable_2, azimuths_2, phi_boundary_2,
             azimuth_branches_2, i_east_2)


**Геометрическое объяснение результатов задания 2:**

*Объясните различие двух направлений и их слияние на границе.
Сопоставьте целевое наклонение с результатом запуска на восток
для каждой площадки. Почему ненулевая скорость вращения площадки
сама по себе не обеспечивает доступность заданной плоскости?*


Чекеры проверяют только численные результаты и значения функций
для заданных исходных данных. Графики иллюстрируют зависимости на сетке.
Эти проверки **не заменяют выводов и объяснений**.

## Что представить

Сдайте этот ноутбук с полными решениями двух заданий:
массовым составом, формулами длительностей и характеристических скоростей,
таблицами и двумя графиками истории, сравнением вариантов разделения;
формулами начальной скорости и доступности плоскости, таблицей азимутов,
графиком двух ветвей и обоснованными ответами на вопросы.
Использование чекеров и сохранение результатов их вызова не требуются.

Перед отправкой перезапустите ядро, выполните ячейки решения по порядку и сохраните
ноутбук с результатами. Название файла: **`hw_7_name_surname.ipynb`**;
вместо `name_surname` укажите свои имя и фамилию латиницей.
